Le but est de comparer les performances des 4 niveaux de corrections avec un seul modèle pour le scénario 3

In [25]:
import pandas as pd
from pathlib import Path

# 1. Dossier courant du notebook
current_dir = Path.cwd()

# 2. Retrouve automatiquement la racine du projet (là où se trouve le dossier 'data')
# Remonte l'arborescence jusqu'à trouver le dossier 'data'
root_dir = next(
    p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists()
)

# 3. Chemin vers le fichier
file_path = root_dir / "data" / "creditcard_pret_ingestion.csv"

# 4. Chargement du CSV
df_niveau0 = pd.read_csv(file_path)
df_niveau0 = df_niveau0[df_niveau0["BILL_AMT1"] > 0]

In [26]:
y_0 = df_niveau0['dpnm']
X_0 = df_niveau0.drop('dpnm', axis=1)

In [27]:
import pandas as pd
from pathlib import Path

# 1. Dossier courant du notebook
current_dir = Path.cwd()

# 2. Retrouve automatiquement la racine du projet (là où se trouve le dossier 'data')
# Remonte l'arborescence jusqu'à trouver le dossier 'data'
root_dir = next(
    p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists()
)

# 3. Chemin vers le fichier
file_path = root_dir / "data" / "cleaned1_creditcard.csv"

# 4. Chargement du CSV
df_niveau1 = pd.read_csv(file_path)
df_niveau1 = df_niveau1[df_niveau1["BILL_AMT1"] > 0]

In [28]:
y_1 = df_niveau1['dpnm']
X_1 = df_niveau1.drop('dpnm', axis=1)

In [29]:
import pandas as pd
from pathlib import Path

# 1. Dossier courant du notebook
current_dir = Path.cwd()

# 2. Retrouve automatiquement la racine du projet (là où se trouve le dossier 'data')
# Remonte l'arborescence jusqu'à trouver le dossier 'data'
root_dir = next(
    p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists()
)

# 3. Chemin vers le fichier
file_path = root_dir / "data" / "cleaned2_creditcard.csv"

# 4. Chargement du CSV
df_niveau2 = pd.read_csv(file_path)
df_niveau2 = df_niveau2[df_niveau2["BILL_AMT1"] > 0]

In [30]:
y_2 = df_niveau2['dpnm']
X_2 = df_niveau2.drop('dpnm', axis=1)

In [31]:
import pandas as pd
from pathlib import Path

# 1. Dossier courant du notebook
current_dir = Path.cwd()

# 2. Retrouve automatiquement la racine du projet (là où se trouve le dossier 'data')
# Remonte l'arborescence jusqu'à trouver le dossier 'data'
root_dir = next(
    p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists()
)

# 3. Chemin vers le fichier
file_path = root_dir / "data" / "cleaned3_creditcard.csv"

# 4. Chargement du CSV
df_niveau3 = pd.read_csv(file_path)
df_niveau3 = df_niveau3[df_niveau3["BILL_AMT1"] > 0]

In [32]:
y_3 = df_niveau3['dpnm']
X_3 = df_niveau3.drop('dpnm', axis=1)

In [33]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# 1. Définition des colonnes
# Séparation des colonnes numériques et catégorielles
num_col = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
cat_col = ['SEX', 'EDUCATION', 'MARRIAGE']
ord_col = ['PAY_1','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']

# 2. Préprocesseur standardisé
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_col),
        (
            "nom",
            OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
            cat_col,
        ),
        (
            "ord",
            OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1
            ),
            ord_col,
        ),
    ]
)

# 3. Pipeline Random Forest avec hyperparamètres FIGÉS
rf_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        min_samples_leaf=30,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
)

# 4. Dictionnaire de tes 4 jeux de données (Scénario S3 appliqués sur L0, L1, L2, L3)
# -> Ajuste les noms de variables si nécessaire (ex: X_L0, y_L0)
datasets = {
    "Niveau 0": (X_0, y_0),
    "Niveau 1": (X_1, y_1),
    "Niveau 2": (X_2, y_2),
    "Niveau 3": (X_3, y_3),
}

# 5. Validation croisée Stratifiée (5-Fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

print(
    "=== EVALUATION DE LA QUALITÉ DES 4 NIVEAUX (RANDOM FOREST FIGÉ - SCÉNARIO S3) ===\n"
)

for level_name, (X, y) in datasets.items():
    # Validation croisée avec score Train & Val pour détecter l'overfitting
    cv_res = cross_validate(
        rf_pipeline,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        return_train_score=True,
        n_jobs=-1,
    )

    train_score = np.mean(cv_res["train_score"])
    val_score = np.mean(cv_res["test_score"])
    std_val = np.std(cv_res["test_score"])

    results.append({
        "Niveau de Nettoyage": level_name,
        "Lignes": len(X),
        "ROC AUC Train": round(train_score, 4),
        "ROC AUC Val": round(val_score, 4),
        "Écart (Overfit)": round(train_score - val_score, 4),
        "Std Val (Stabilité)": round(std_val, 4),
    })

# 6. Synthèse sous forme de tableau
df_comparison = pd.DataFrame(results)
print(df_comparison.to_string(index=False))

=== EVALUATION DE LA QUALITÉ DES 4 NIVEAUX (RANDOM FOREST FIGÉ - SCÉNARIO S3) ===

Niveau de Nettoyage  Lignes  ROC AUC Train  ROC AUC Val  Écart (Overfit)  Std Val (Stabilité)
           Niveau 0   27402         0.8173       0.7886           0.0287               0.0044
           Niveau 1   27398         0.8171       0.7889           0.0283               0.0045
           Niveau 2   27398         0.8172       0.7889           0.0283               0.0047
           Niveau 3   27398         0.8160       0.7874           0.0287               0.0046


In [34]:
from sklearn.metrics import fbeta_score, make_scorer

# 1. Création du scorer F2
f2_scorer = make_scorer(fbeta_score, beta=2)

# 2. Dictionnaire de scoring pour tout calculer d'un coup
scoring = {"roc_auc": "roc_auc", "f2": f2_scorer}

results = []

for level_name, (X, y) in datasets.items():
    cv_res = cross_validate(
        rf_pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1,
    )

    results.append({
        "Niveau de Nettoyage": level_name,
        "Lignes": len(X),
        "ROC AUC Val": round(np.mean(cv_res["test_roc_auc"]), 4),
        "F2 Train": round(np.mean(cv_res["train_f2"]), 4),
        "F2 Val": round(np.mean(cv_res["test_f2"]), 4),
        "Écart F2 (Overfit)": round(
            np.mean(cv_res["train_f2"]) - np.mean(cv_res["test_f2"]), 4
        ),
        "Std F2 Val": round(np.std(cv_res["test_f2"]), 4),
    })

df_comparison = pd.DataFrame(results)
print(df_comparison.to_string(index=False))

Niveau de Nettoyage  Lignes  ROC AUC Val  F2 Train  F2 Val  Écart F2 (Overfit)  Std F2 Val
           Niveau 0   27402       0.7886    0.6042  0.5843              0.0199      0.0102
           Niveau 1   27398       0.7889    0.6042  0.5824              0.0217      0.0093
           Niveau 2   27398       0.7889    0.6043  0.5831              0.0212      0.0087
           Niveau 3   27398       0.7874    0.6026  0.5821              0.0205      0.0088
